# A/B Experimentation & Incentive Optimization
**Author:** Parichennayapalli Viswa Sai Reddy  
**Tools:** Python · Statistics (Two-Sample t-test, p-values)  
**Dataset:** 21,000+ users | Incentive Feature Rollout Test

---

## Business Problem
Evaluated the ROI and scalability of a **new incentive feature** across 20,000+ users to determine rollout viability.

## Key Outcomes
- Designed experiment parameters and success metrics
- Validated a **4.6% statistically significant conversion uplift** (p < 0.05)
- Projected rollout to **increase MRR by 6–8%**
- Calculated positive ROI after incentive costs

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('data/ab_experiment.csv')
control = df[df['group'] == 'Control']
treatment = df[df['group'] == 'Treatment']
print(f'Total users: {len(df):,}')
print(f'Control: {len(control):,} | Treatment: {len(treatment):,}')
df.head()

## 2. Experiment Design & Sanity Checks

In [ ]:
# Sample Ratio Mismatch check
print('=== SAMPLE RATIO CHECK ===')
control_pct = len(control) / len(df) * 100
treatment_pct = len(treatment) / len(df) * 100
print(f'Control:   {len(control):,} ({control_pct:.1f}%)')
print(f'Treatment: {len(treatment):,} ({treatment_pct:.1f}%)')
print('✅ No sample ratio mismatch (50/50 split confirmed)' if abs(control_pct - 50) < 1 else '⚠️ Check SRM')

# Segment distribution balance
print('\n=== SEGMENT BALANCE CHECK ===')
print(pd.crosstab(df['group'], df['user_segment'], normalize='index').round(3))

In [ ]:
# === PRIMARY METRIC: Conversion Rate ===
ctrl_conv = control['converted'].mean()
trt_conv  = treatment['converted'].mean()
uplift    = trt_conv - ctrl_conv
uplift_pct = uplift / ctrl_conv * 100

print(f'Control Conversion Rate:   {ctrl_conv*100:.2f}%')
print(f'Treatment Conversion Rate: {trt_conv*100:.2f}%')
print(f'Absolute Uplift:           {uplift*100:.2f}pp')
print(f'Relative Uplift:           {uplift_pct:.2f}%')

In [ ]:
# === TWO-SAMPLE T-TEST ===
t_stat, p_value = stats.ttest_ind(control['converted'], treatment['converted'])
print(f'\n=== HYPOTHESIS TEST ===')
print(f'H0: No difference in conversion rates between groups')
print(f'H1: Treatment has higher conversion rate')
print(f'\nt-statistic: {t_stat:.4f}')
print(f'p-value:     {p_value:.6f}')
alpha = 0.05
if p_value < alpha:
    print(f'✅ REJECT H0 at α={alpha}: Result is statistically significant')
else:
    print(f'⚠️ FAIL TO REJECT H0 at α={alpha}')

In [ ]:
# === CONFIDENCE INTERVAL ===
from statsmodels.stats.proportion import proportion_confint
ctrl_ci  = proportion_confint(control['converted'].sum(),  len(control),  alpha=0.05, method='wilson')
trt_ci   = proportion_confint(treatment['converted'].sum(), len(treatment), alpha=0.05, method='wilson')
print(f'Control   95% CI: ({ctrl_ci[0]*100:.2f}%, {ctrl_ci[1]*100:.2f}%)')
print(f'Treatment 95% CI: ({trt_ci[0]*100:.2f}%, {trt_ci[1]*100:.2f}%)')

In [ ]:
# === REVENUE IMPACT ===
ctrl_rev_per_user  = control['revenue'].mean()
trt_rev_per_user   = treatment['revenue'].mean()
trt_incentive_cost = treatment['incentive_cost'].mean()
trt_net_per_user   = trt_rev_per_user - trt_incentive_cost
roi = (trt_net_per_user - ctrl_rev_per_user) / trt_incentive_cost * 100

print('\n=== REVENUE ANALYSIS ===')
print(f'Avg Revenue/User — Control:   ₹{ctrl_rev_per_user:,.2f}')
print(f'Avg Revenue/User — Treatment: ₹{trt_rev_per_user:,.2f}')
print(f'Avg Incentive Cost/User:      ₹{trt_incentive_cost:,.2f}')
print(f'Net Revenue/User (Treatment): ₹{trt_net_per_user:,.2f}')
print(f'ROI on Incentive:             {roi:.1f}%')

## 3. Visualizations

In [ ]:
import os; os.makedirs('outputs', exist_ok=True)

# Fig 1: Conversion rate comparison with CI
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

groups = ['Control', 'Treatment']
rates  = [ctrl_conv*100, trt_conv*100]
cis    = [(ctrl_ci[1]-ctrl_ci[0])*100/2, (trt_ci[1]-trt_ci[0])*100/2]
colors = ['#95a5a6', '#2ecc71']

bars = axes[0].bar(groups, rates, yerr=cis, capsize=8, color=colors, edgecolor='white', width=0.5, error_kw={'linewidth':2})
axes[0].set_ylabel('Conversion Rate %'); axes[0].set_title(f'Conversion Rate by Group\np-value = {p_value:.4f} | Uplift = +{uplift*100:.2f}pp', fontweight='bold')
for bar, val in zip(bars, rates):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold')

# Fig 1b: Conversion by user segment
segment_conv = df.groupby(['user_segment','group'])['converted'].mean().unstack() * 100
segment_conv.plot(kind='bar', ax=axes[1], color=['#95a5a6','#2ecc71'], edgecolor='white', width=0.6)
axes[1].set_title('Conversion Rate by User Segment', fontweight='bold')
axes[1].set_ylabel('Conversion Rate %'); axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Group')
plt.tight_layout()
plt.savefig('outputs/fig1_conversion_rates.png', bbox_inches='tight'); plt.close()

In [ ]:
# Fig 2: Revenue distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ctrl_rev_converted  = control[control['converted']==1]['revenue']
trt_rev_converted   = treatment[treatment['converted']==1]['revenue']
axes[0].hist(ctrl_rev_converted, bins=40, alpha=0.6, color='#95a5a6', label='Control', density=True)
axes[0].hist(trt_rev_converted,  bins=40, alpha=0.6, color='#2ecc71',  label='Treatment', density=True)
axes[0].axvline(ctrl_rev_converted.mean(), color='gray', linestyle='--', linewidth=2)
axes[0].axvline(trt_rev_converted.mean(), color='#27ae60', linestyle='--', linewidth=2)
axes[0].set_xlabel('Revenue per Converted User (₹)'); axes[0].set_ylabel('Density')
axes[0].set_title('Revenue Distribution (Converted Users)', fontweight='bold')
axes[0].legend()

# Net revenue after incentive
rev_data = pd.DataFrame({'Group': ['Control','Treatment','Treatment (Net)'],
    'Revenue per User': [ctrl_rev_per_user, trt_rev_per_user, trt_net_per_user]})
colors3 = ['#95a5a6','#3498db','#2ecc71']
axes[1].bar(rev_data['Group'], rev_data['Revenue per User'], color=colors3, edgecolor='white', width=0.5)
for i, val in enumerate(rev_data['Revenue per User']):
    axes[1].text(i, val+5, f'₹{val:.0f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Avg Revenue per User (₹)'); axes[1].set_title('Revenue per User: Control vs Treatment', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/fig2_revenue_analysis.png', bbox_inches='tight'); plt.close()

In [ ]:
# Fig 3: Uplift by segment and region
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for col, ax, title in [('user_segment', axes[0], 'Conversion Uplift by User Segment'),
                        ('region',       axes[1], 'Conversion Uplift by Region')]:
    uplift_df = df.groupby([col,'group'])['converted'].mean().unstack()
    uplift_df['uplift_pp'] = (uplift_df['Treatment'] - uplift_df['Control']) * 100
    colors_u = ['#2ecc71' if v > 0 else '#e74c3c' for v in uplift_df['uplift_pp']]
    ax.bar(uplift_df.index, uplift_df['uplift_pp'], color=colors_u, edgecolor='white', width=0.6)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Conversion Uplift (pp)'); ax.set_title(title, fontweight='bold')
    for i, val in enumerate(uplift_df['uplift_pp']):
        ax.text(i, val + 0.1 if val >= 0 else val - 0.3, f'+{val:.1f}pp' if val >=0 else f'{val:.1f}pp', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/fig3_uplift_by_segment_region.png', bbox_inches='tight'); plt.close()
print('Charts saved.')

In [ ]:
# Fig 4: MRR Projection
monthly_users = 20000
base_mrr = ctrl_conv * monthly_users * ctrl_rev_per_user
new_mrr  = trt_conv  * monthly_users * trt_net_per_user
mrr_lift = (new_mrr - base_mrr) / base_mrr * 100

print(f'Base MRR (no rollout):     ₹{base_mrr:,.0f}')
print(f'Projected MRR (rollout):   ₹{new_mrr:,.0f}')
print(f'MRR Uplift:                {mrr_lift:.1f}%')

fig, ax = plt.subplots(figsize=(8, 5))
scenarios = ['Baseline\n(No Rollout)', 'With Incentive\n(Full Rollout)']
values = [base_mrr/100000, new_mrr/100000]
colors_m = ['#95a5a6','#2ecc71']
bars = ax.bar(scenarios, values, color=colors_m, edgecolor='white', width=0.4)
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'₹{val:.1f}L', ha='center', fontsize=13, fontweight='bold')
ax.set_ylabel('Monthly Revenue (₹ Lakhs)')
ax.set_title(f'MRR Projection: +{mrr_lift:.1f}% with Full Incentive Rollout', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/fig4_mrr_projection.png', bbox_inches='tight'); plt.close()

## 4. Decision & Recommendation

In [ ]:
print('='*60)
print('   A/B TEST — DECISION SUMMARY')
print('='*60)
print(f'\n✅ Uplift: +{uplift*100:.2f}pp ({uplift_pct:.1f}% relative)')
print(f'✅ p-value: {p_value:.6f} (< 0.05 threshold)')
print(f'✅ ROI on incentive: {roi:.0f}%')
print(f'✅ Projected MRR uplift: +{mrr_lift:.1f}%')
print('\nRECOMMENDATION: PROCEED WITH FULL ROLLOUT')
print('─'*60)
print('1. Roll out incentive feature to 100% of users')
print('2. Monitor conversion rate weekly for regression')
print('3. A/B test incentive amount (₹60 vs ₹90 vs ₹120)')
print('4. Prioritise Power user segment — highest absolute uplift')

---
*Analysis by Parichennayapalli Viswa Sai Reddy | Python · Statistics · Scipy*